In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import PartialDependenceDisplay
from pygam import LinearGAM, s
from matplotlib.cm import ScalarMappable
import matplotlib as mpl
from sklearn.linear_model import LinearRegression

# =======================
# 1. 读数据
# =======================
file_path = r"D:\研究\ABF新\8.13ML\ML\生产率及解释变量合并2.xlsx"
df = pd.read_excel(file_path, sheet_name=0)
y_col = "gProtein_per_kg"
# 调整顺序：Feed_Yield, Ruminant_Share, T
x_cols = ["Feed_Yield", "Ruminant_Share", "T"]

# 只保留需要的列并去掉 NaN
data = df[x_cols + [y_col]].dropna().copy()

# =======================
# 2. IQR 异常值剔除
# =======================
def iqr_mask(series, k=1.5):
    """根据 IQR 返回一个布尔掩码：True = 保留, False = 异常值"""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return (series >= lower) & (series <= upper)

mask = np.ones(len(data), dtype=bool)
for col in x_cols + [y_col]:
    mask &= iqr_mask(data[col])
data_clean = data.loc[mask].copy()
print(f"原始样本数: {len(data)}, 去除 IQR 异常值后样本数: {len(data_clean)}")

X = data_clean[x_cols]
y = data_clean[y_col].values

# =======================
# 3. 训练 PDP 用的模型
# =======================
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X, y)

# =======================
# 4. 全局绘图参数（增大字号）
# =======================
plt.rcParams["font.sans-serif"] = ["SimHei", "Arial"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.size"] = 16
plt.rcParams["axes.titlesize"] = 18
plt.rcParams["axes.labelsize"] = 16
plt.rcParams["xtick.labelsize"] = 14
plt.rcParams["ytick.labelsize"] = 14
plt.rcParams["legend.fontsize"] = 14

# 变量标签映射
x_labels = {
    "Feed_Yield": "Feed Yield",
    "Ruminant_Share": "Ruminant%",
    "T": "T"
}
y_label = "gProtein/kg"

# =======================
# 5. 创建子图布局
# =======================
fig = plt.figure(figsize=(12, 8))
gs = fig.add_gridspec(3, 3, height_ratios=[1, 1, 0.08], hspace=0.35, wspace=0.25)

# 上两行用于子图 a–f
axes = []
for i in range(6):
    row = i // 3
    col = i % 3
    axes.append(fig.add_subplot(gs[row, col]))

# 第三行用于色条（占满整行）
cbar_ax1 = fig.add_subplot(gs[2, 0])
cbar_ax2 = fig.add_subplot(gs[2, 1])
cbar_ax3 = fig.add_subplot(gs[2, 2])

# 子图标签
subplot_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]

# =======================
# 6. 顶行：a, b, c 响应曲线（GAM 曲线 + 置信区间 + 线性趋势线）
# =======================
for i, var in enumerate(x_cols):
    ax = axes[i]
    x = data_clean[var].values

    # (1) GAM 拟合线（基于完整样本）
    gam = LinearGAM(s(0)).fit(x.reshape(-1, 1), y)
    xx = np.linspace(np.nanpercentile(x, 2), np.nanpercentile(x, 98), 500)
    yy = gam.predict(xx.reshape(-1, 1))
    
    # 获取置信区间
    confi = gam.confidence_intervals(xx.reshape(-1, 1), width=0.95)
    
    # 绘制置信区间（使用散点同款颜色 #90B4CF）
    ax.fill_between(xx, confi[:, 0], confi[:, 1], 
                     color="#90B4CF", alpha=0.4, 
                     label="95% CI")

    # (2) GAM 曲线
    gam_line = ax.plot(
        xx, yy, linestyle="-", linewidth=2.5,
        color="#337BAC", label="GAM curve"
    )

    # (3) 添加线性趋势线
    # 使用原始数据（不采样）进行线性拟合
    x_linear = x.reshape(-1, 1)
    y_linear = y
    
    lin_reg = LinearRegression()
    lin_reg.fit(x_linear, y_linear)
    
    # 生成线性预测值
    x_range = np.linspace(x.min(), x.max(), 2)  # 只需要起点和终点
    y_linear_pred = lin_reg.predict(x_range.reshape(-1, 1))
    
    # 绘制线性趋势线
    linear_line = ax.plot(
        x_range, y_linear_pred, 
        linestyle="--", linewidth=2.0,
        color="#FF6B6B", label="Linear trend"
    )

    # 坐标轴标签（使用修改后的标签）
    ax.set_xlabel(x_labels.get(var, var))
    ax.set_ylabel(y_label)

    # 只在 (a) 子图中添加图例
    if i == 0:
        ax.legend(frameon=True, loc='upper right')

    # 添加子图编号
    ax.text(
        0.02, 0.98, subplot_labels[i],
        transform=ax.transAxes,
        fontsize=14, fontweight='bold',
        va='top', ha='left'
    )

    # 调整刻度字号
    ax.tick_params(axis='both', labelsize=14)

# =======================
# 7. 底行：d, e, f PDP 交互作用（去掉标题 & 去掉下划线）
# =======================
pdp_plots = []

pdp_pairs = [
    ("Feed_Yield", "Ruminant_Share"),
    ("Feed_Yield", "T"),
    ("Ruminant_Share", "T")
]

for i, (feat1, feat2) in enumerate(pdp_pairs):
    # 先把 axes[i+3] 交给 PDP 画
    display = PartialDependenceDisplay.from_estimator(
        rf, X, [(feat1, feat2)],
        ax=axes[i + 3], kind="average"
    )
    pdp_plots.append(display)

    # 从 display 中拿真正的轴
    if hasattr(display, "axes_"):
        pdp_ax = np.ravel(display.axes_)[0]
    else:
        pdp_ax = axes[i + 3]

    # 改成无下划线的标签
    pdp_ax.set_xlabel(x_labels[feat1])
    pdp_ax.set_ylabel(x_labels[feat2])

    # 去掉自动生成的标题
    pdp_ax.set_title("")

    # 子图编号
    pdp_ax.text(
        0.02, 0.98, subplot_labels[i + 3],
        transform=pdp_ax.transAxes,
        fontsize=14, fontweight='bold',
        va='top', ha='left'
    )

    pdp_ax.tick_params(axis='both', labelsize=11)

# =======================
# 8. 添加 PDP 色条（并调窄色带厚度）
# =======================
for i, (display, cax) in enumerate(zip(pdp_plots, [cbar_ax1, cbar_ax2, cbar_ax3])):
    # 从 PDP 图中提取数据
    pdp_results = display.pd_results[0]
    pdp_values = pdp_results["average"]

    # 创建色条
    norm = mpl.colors.Normalize(vmin=pdp_values.min(), vmax=pdp_values.max())
    sm = ScalarMappable(norm=norm, cmap='viridis')
    sm.set_array([])

    cbar = plt.colorbar(sm, cax=cax, orientation='horizontal')
    cbar.set_label('Partial dependence', fontsize=14)
    cbar.ax.tick_params(labelsize=10)

# 先整体布局，再手动压缩色带厚度（长度保持不变）
plt.tight_layout()

for cax in [cbar_ax1, cbar_ax2, cbar_ax3]:
    pos = cax.get_position()
    new_height = pos.height * 0.5  # 厚度减半
    # 调整色带与图的距离：增加0.02让色带离图更远
    new_y0 = pos.y0 + 0.02
    cax.set_position([pos.x0, new_y0, pos.width, new_height])


# 先保存
plt.savefig(
    r"D:\研究\ABF新\论文的图\response_PDP.png",
    dpi=300,
    bbox_inches="tight"
)

# 再显示
plt.show()

In [ ]:
import os
import math
import numpy as np
import pandas as pd

# ========== 用户参数 ==========
EXCEL_PATH = r"D:\研究\ABF新\5.23 ABF\生产率总_贸易_带区域.xlsx"
SHEET_NAME = 0  # 如果不是第一个工作表, 改成对应 sheet 名称或索引
EPS = 1e-10  # 处理 0 值的极小数
TOPK = 5  # 剔除实验时的“前 K 大国家”（按长期饲料 F）
OUTDIR = r"D:\研究\ABF新\5.23 ABF\LMDI蛋白"  # ← 修改后的输出目录
os.makedirs(OUTDIR, exist_ok=True)  # 确保目录存在

# ========== 基础函数 ==========
def log_mean(x, y, eps=1e-10):
    """对数平均 L(x,y)；含相等与非正数的稳健处理"""
    try:
        x = float(x); y = float(y)
    except Exception:
        x, y = eps, eps
    if x <= 0 and y <= 0:
        return eps
    if abs(x - y) < 1e-15:
        return max(x, eps)  # L(x,x)=x
    x = max(x, eps); y = max(y, eps)
    return (x - y) / (math.log(x) - math.log(y))

def lmdi_endpoint(df, start_year, end_year):
    """
    端点式 LMDI（蛋白生产率口径）：  
    先对强度 I=F/O 分解（结构=产出份额 q，强度=单位产出投入），
    再取负得到生产率 P=1/I 的分解。
    其中 O = gProtein（全量蛋白，单位随源数据：一般为 g），F = Total_kg（饲料质量，kg）
    输入 df：唯一 Code-Year 的长表，含列 Code, Year, Area, Region, gProtein(=O), Total_kg(=F)
    返回：每国结构/效率/合计贡献（针对 Δln P）
    """
    d0 = df[df["Year"] == start_year].copy()
    dT = df[df["Year"] == end_year].copy()

    # 聚齐国家集合（缺失者补 0）
    all_countries = sorted(set(d0["Code"]).union(set(dT["Code"])))
    d0 = d0.set_index("Code").reindex(all_countries)
    dT = dT.set_index("Code").reindex(all_countries)

    # 填缺省
    for col in ["gProtein", "Total_kg"]:
        d0[col] = pd.to_numeric(d0[col], errors="coerce").fillna(0.0)
        dT[col] = pd.to_numeric(dT[col], errors="coerce").fillna(0.0)
    for col in ["Area", "Region"]:
        d0[col] = d0[col].fillna("")
        dT[col] = dT[col].fillna("")

    O0 = max(d0["gProtein"].sum(), EPS)
    OT = max(dT["gProtein"].sum(), EPS)
    F0 = max(d0["Total_kg"].sum(), EPS)
    FT = max(dT["Total_kg"].sum(), EPS)

    # 权重 Wi = L(Fi_T, Fi_0) / L(F_T, F_0)
    denom = log_mean(FT, F0, EPS)
    if denom <= 0:
        denom = EPS

    out_rows = []
    for code in all_countries:
        area = dT.loc[code, "Area"] if dT.loc[code, "Area"] else d0.loc[code, "Area"]
        region = dT.loc[code, "Region"] if dT.loc[code, "Region"] else d0.loc[code, "Region"]
        Oi0 = max(float(d0.loc[code, "gProtein"]), EPS)
        OiT = max(float(dT.loc[code, "gProtein"]), EPS)
        Fi0 = max(float(d0.loc[code, "Total_kg"]), EPS)
        FiT = max(float(dT.loc[code, "Total_kg"]), EPS)

        qi0 = Oi0 / O0
        qiT = OiT / OT
        Ii0 = Fi0 / Oi0
        IiT = FiT / OiT

        Wi = log_mean(FiT, Fi0, EPS) / denom

        # 针对 Δln P 的贡献：取负号（P=1/I）
        C_str = - Wi * (math.log(qiT) - math.log(qi0))  # 结构效应（产出份额）
        C_eff = - Wi * (math.log(IiT) - math.log(Ii0))  # 效率效应（单位产出投入）
        C_tot = C_str + C_eff

        out_rows.append({
            "Code": code, "Area": area, "Region": region,
            "C_str": C_str, "C_eff": C_eff, "C_total": C_tot
        })

    res = pd.DataFrame(out_rows)
    # 校验零残差：各国合计 ≈ Δln P
    delta_lnP = math.log( (OT/FT) / (O0/F0) )
    res_total = res[["C_str","C_eff","C_total"]].sum()
    res.attrs["delta_lnP_theory"] = delta_lnP
    res.attrs["delta_lnP_sum"] = float(res_total["C_total"])
    res.attrs["start_year"] = start_year
    res.attrs["end_year"] = end_year
    return res

# ========== 读取与全局聚合 ==========
print("【步骤A】读取并整理数据...")
raw = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME)  # 统一列名映射（仅保留分析所需列）
df = raw.rename(columns={
    "Area": "Area", "Year": "Year",
    "gProtein": "gProtein",                # 蛋白总量（单位依源数据；常见为 g）
    "Total_kg": "Total_kg",                # 饲料重量（kg）
    "Code": "Code", "Region": "Region"
})  # 基本清洗
for col in ["gProtein", "Total_kg"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)
df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype(int)  # 统一字符串、去空格
for col in ["Area", "Code", "Region"]:
    df[col] = df[col].astype(str).str.strip()
df["gProtein"] = df["gProtein"].clip(lower=0)
df["Total_kg"] = df["Total_kg"].clip(lower=0)

# 聚合处理
map_area = (df.groupby("Code")["Area"]
            .agg(lambda s: s.mode().iat[0] if not s.mode().empty else s.iloc[0]))
map_region = (df.groupby("Code")["Region"]
              .agg(lambda s: s.mode().iat[0] if not s.mode().empty else s.iloc[0]))

df = (df.groupby(["Code", "Year"], as_index=False)
      .agg(gProtein=("gProtein", "sum"), Total_kg=("Total_kg", "sum")))
df["Area"] = df["Code"].map(map_area)
df["Region"] = df["Code"].map(map_region)

# 端点式 LMDI 结果（1980-2009）
start_year = 1980
end_year = 2009
print(f"【步骤C】端点式 LMDI（1980→2009）...")
res_full_1980_2009 = lmdi_endpoint(df, start_year, end_year)

# 端点式 LMDI 结果（2009-2022）
start_peak = 2009
end_peak = int(df["Year"].max())
print(f"【步骤D】端点式 LMDI（2009→2022）...")
res_peak = lmdi_endpoint(df, start_peak, end_peak)

# 端点式 LMDI 结果（1980-2022）
start_full = int(df["Year"].min())
end_full = int(df["Year"].max())
print(f"【步骤E】端点式 LMDI（1980→2022）...")
res_full = lmdi_endpoint(df, start_full, end_full)

# 找出1980-2009贡献最大的前十位国家
top_1980_2009 = res_full_1980_2009.sort_values(by="C_total", ascending=False).head(10)

# 找出2009-2022贡献最大的前十位国家（对于下降趋势，贡献最小的排在前面，即C_total最负的国家）
top_2009_2022 = res_peak.sort_values(by="C_total", ascending=True).head(10)

# 找出1980-2022年贡献绝对值最大的国家
res_full["abs_C_total"] = res_full["C_total"].abs()
top_abs_1980_2022 = res_full.sort_values(by="abs_C_total", ascending=False).head(10)

# 将结果输出到Excel文件的不同sheet
output_path = os.path.join(OUTDIR, "top_contributors_by_period.xlsx")
with pd.ExcelWriter(output_path) as writer:
    top_1980_2009.to_excel(writer, sheet_name="Top_1980_2009", index=False)
    top_2009_2022.to_excel(writer, sheet_name="Top_2009_2022", index=False)
    top_abs_1980_2022.to_excel(writer, sheet_name="Top_Absolute_1980_2022", index=False)

print(f"结果已输出到：{output_path}")

In [ ]:
import pandas as pd

# 读取数据
file_path = "D:\\研究\\ABF新\\5.23 ABF\\LMDI蛋白\\protein_lmdi_chain_yearly.csv"
df = pd.read_csv(file_path)

# 定义目标国家及其代码
target_countries = {
    'IND': 'India',
    'USA': 'USA', 
    'CHN': 'China',
    'ARG': 'Argentina',
    'OWID_USS': 'USSR',
    'RUS': 'Russia',
    'BRA': 'Brazil',
    'BGD': 'Bangladesh',
    'DEU': 'Germany',
    'PAK': 'Pakistan',
    'AUS': 'Australia',
    'VEN': 'Venezuela',
    'TUR': 'Turkey'
}

# 筛选目标国家的数据
selected_data = df[df['Code'].isin(target_countries.keys())].copy()

# 按国家和年份排序
selected_data = selected_data.sort_values(['Code', 'start_year'])

# 设置显示选项，确保显示所有小数
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.6f}'.format)

print("各国逐年LMDI分解结果:")
print("=" * 80)

# 为每个国家打印数据
for code, country_name in target_countries.items():
    country_data = selected_data[selected_data['Code'] == code]
    
    if country_data.empty:
        print(f"\n{country_name} ({code}): 无数据")
        continue
    
    print(f"\n{country_name} ({code}):")
    print("-" * 60)
    print(f"{'Year':>6} {'C_str':>12} {'C_eff':>12} {'C_total':>12}")
    print("-" * 60)
    
    for _, row in country_data.iterrows():
        print(f"{row['start_year']:>6} {row['C_str']:>12.6f} {row['C_eff']:>12.6f} {row['C_total']:>12.6f}")

print("\n" + "=" * 80)
print("数据汇总完成！")

# 可选：保存到Excel文件以便进一步分析
output_excel_path = "D:\\研究\\ABF新\\论文的图\\selected_countries_lmdi_results.xlsx"
try:
    # 为每个国家创建单独的工作表
    with pd.ExcelWriter(output_excel_path, engine='openpyxl') as writer:
        for code, country_name in target_countries.items():
            country_data = selected_data[selected_data['Code'] == code]
            if not country_data.empty:
                # 选择需要的列并重命名
                output_df = country_data[['start_year', 'C_str', 'C_eff', 'C_total']].copy()
                output_df = output_df.rename(columns={
                    'start_year': 'Year',
                    'C_str': 'Structure_Effect',
                    'C_eff': 'Efficiency_Effect', 
                    'C_total': 'Total_Effect'
                })
                output_df.to_excel(writer, sheet_name=country_name[:31], index=False)  # 工作表名最多31字符
    
    print(f"\n数据已保存到Excel文件: {output_excel_path}")
    
except Exception as e:
    print(f"\n保存Excel文件时出错: {e}")

# 可选：显示一些统计信息
print("\n基本统计信息:")
print("-" * 40)
for code, country_name in target_countries.items():
    country_data = selected_data[selected_data['Code'] == code]
    if not country_data.empty:
        total_effect = country_data['C_total'].sum()
        str_effect = country_data['C_str'].sum()
        eff_effect = country_data['C_eff'].sum()
        years = len(country_data)
        print(f"{country_name:15} | 年份数: {years:2} | 总效应: {total_effect:9.6f} | 结构效应: {str_effect:9.6f} | 效率效应: {eff_effect:9.6f}")